# AWS CLI v2

AWS CLI v2 is not on PyPI; install it with AWS's installer (the PyPI project `awscliv2` is an unrelated wrapper). These cells drive the `aws` command on `PATH` through `subprocess`, so they work with any AWS CLI v2 install.

Because `pip install awscliv2` does not provide the `aws` command, this notebook is tagged `skip-by-default`: install AWS CLI v2 first, then run it with `--with all`. The S3 cells read the public `noaa-goes16` bucket anonymously (`--no-sign-request`) and need the `network` capability.

In [ ]:
import shutil
import subprocess

launcher = shutil.which("aws")
assert launcher, "no `aws` on PATH"
completed = subprocess.run([launcher, "--version"], capture_output=True, text=True, timeout=120)
assert completed.returncode == 0, f"aws --version failed rc={completed.returncode}: {completed.stderr[:300]}"
assert "aws-cli/2." in completed.stdout, f"expected AWS CLI v2, got: {completed.stdout.strip()}"
print("launcher:", launcher)
print(completed.stdout.strip())

In [ ]:
import json
import subprocess

completed = subprocess.run(
    ["aws", "s3api", "list-objects-v2",
     "--bucket", "noaa-goes16", "--no-sign-request", "--max-items", "2"],
    capture_output=True, text=True, timeout=300,
)
assert completed.returncode == 0, (
    f"rc={completed.returncode}\nstderr: {completed.stderr[-400:]}"
)
payload = json.loads(completed.stdout)
assert payload.get("Contents"), f"no Contents in response: {list(payload)}"
assert payload["Contents"][0].get("Key"), "first object has no Key"
print(f"parsed {len(payload['Contents'])} object(s); first key:")
print(" ", payload["Contents"][0]["Key"][:90])

In [ ]:
import json
import subprocess
import tempfile

BUCKET = "noaa-goes16"
PREFIX = "ABI-L1b-RadC/2000/001/12/"

def aws(*arguments, timeout=900):
    completed = subprocess.run(["aws", *arguments], capture_output=True, text=True, timeout=timeout)
    assert completed.returncode == 0, f"`aws {' '.join(arguments)}` rc={completed.returncode}\n{completed.stderr[-500:]}"
    return completed.stdout

def keys_of(payload):
    return [entry["Key"] for entry in json.loads(payload).get("Contents", [])]

# One request per page, default page size.
single = keys_of(aws("s3api", "list-objects-v2", "--bucket", BUCKET, "--prefix", PREFIX, "--no-sign-request"))
# THE FLOOR, and it comes first on purpose: an empty or vanished prefix returns [] from both calls and
# would sail through the set-equality below. `0 == 0` is not a passing paginator test.
assert len(single) >= 100, f"only {len(single)} keys under {PREFIX} -- expected ~272; the prefix moved or the listing broke"

# Same listing, forced through ~28 pages. The CLI must merge them into the identical set.
paged = keys_of(aws("s3api", "list-objects-v2", "--bucket", BUCKET, "--prefix", PREFIX, "--no-sign-request",
                    "--page-size", "10"))
assert len(paged) == len(single) and set(paged) == set(single), (
    f"paginated listing disagrees with the single-page one: {len(paged)} vs {len(single)} keys, "
    f"{len(set(single) ^ set(paged))} differing. The paginator dropped or duplicated a page."
)
print(f"  pagination:    {len(single)} keys, identical across default and --page-size 10  ok")

# The control: --max-items must TRUNCATE and hand back a resumption token. If it returned everything,
# the assert above would be testing nothing about paging.
truncated = json.loads(aws("s3api", "list-objects-v2", "--bucket", BUCKET, "--prefix", PREFIX,
                           "--no-sign-request", "--max-items", "5"))
assert len(truncated["Contents"]) == 5, f"--max-items 5 returned {len(truncated['Contents'])} keys"
assert truncated.get("NextToken"), "--max-items truncated the result but returned no NextToken"
print("  control:       --max-items 5 truncates and returns a NextToken  ok")

# Recursion: `s3 sync` walks the prefix tree and diffs it against a local dir. `--dryrun` does the whole
# listing + comparator pass and transfers nothing, so recursion is proven inside the size budget.
with tempfile.TemporaryDirectory() as empty_path:
    planned = [line for line in aws("s3", "sync", f"s3://{BUCKET}/ABI-L1b-RadC/2000/001/", empty_path,
                                    "--no-sign-request", "--dryrun").splitlines() if line.startswith("(dryrun)")]
assert len(planned) >= 100, f"sync --dryrun planned only {len(planned)} downloads -- recursion did not descend"
assert any("/12/" in line for line in planned), "sync did not descend into the 12/ sub-prefix"
print(f"  recursion:     sync --dryrun planned {len(planned)} transfers across sub-prefixes  ok")

In [ ]:
# RANGED (MULTIPART) DOWNLOAD, proven by md5 against a single-GET control.
#
# s3transfer splits a download into concurrent ranged GETs once the object exceeds `multipart_threshold`,
# then reassembles them. Reassembly in the wrong order, or a dropped range, produces a file of the RIGHT
# SIZE and the wrong content -- so a size check alone certifies nothing. The A/B md5 is the real test.
#
# Config goes in a TEMP `AWS_CONFIG_FILE`, never `aws configure set`: that would mutate the invoking
# user's real `~/.aws/config`.
import hashlib
import json
import os
import pathlib
import re
import subprocess
import tempfile

BUCKET = "noaa-goes16"
KEY = ("ABI-L1b-RadC/2000/001/12/"
       "OR_ABI-L1b-RadC-M3C01_G16_s20000011200000_e20000011200000_c20170751219598.nc")

head = json.loads(subprocess.run(
    ["aws", "s3api", "head-object", "--bucket", BUCKET, "--key", KEY, "--no-sign-request"],
    capture_output=True, text=True, timeout=300, check=True).stdout)
expected_length = head["ContentLength"]
assert expected_length > 3_000_000, f"object is only {expected_length} B -- too small to force a split"

def download(directory_path, *, threshold, chunksize):
    config_path = pathlib.Path(directory_path) / "aws_config"
    config_path.write_text(
        f"[default]\ns3 =\n  multipart_threshold = {threshold}\n  multipart_chunksize = {chunksize}\n"
    )
    target_path = pathlib.Path(directory_path) / "object.nc"
    environment = {**os.environ, "AWS_CONFIG_FILE": str(config_path)}
    environment.pop("AWS_PROFILE", None)
    environment.pop("AWS_DEFAULT_PROFILE", None)
    completed = subprocess.run(
        ["aws", "s3", "cp", f"s3://{BUCKET}/{KEY}", str(target_path), "--no-sign-request", "--debug"],
        capture_output=True, text=True, timeout=900, env=environment,
    )
    assert completed.returncode == 0, f"cp rc={completed.returncode}\n{completed.stderr[-500:]}"
    payload = target_path.read_bytes()
    # Count the DISTINCT Range headers actually put on the wire. Without this the md5 comparison below
    # is vacuous: if the config never took effect both arms would be plain GETs of the same object and
    # would trivially agree. The trailing range is open-ended (`bytes=3145728-`), so match that shape too.
    ranges = set(re.findall(r"bytes=\d+-\d*", completed.stderr))
    return len(payload), hashlib.md5(payload).hexdigest(), len(ranges)

with tempfile.TemporaryDirectory() as whole_path:
    # Threshold far above the object size -> ONE plain GET. The control.
    whole_length, whole_md5, whole_ranges = download(whole_path, threshold="512MB", chunksize="512MB")
with tempfile.TemporaryDirectory() as ranged_path:
    # 1 MB chunks over a 3.4 MB object -> four concurrent ranged GETs, reassembled locally.
    ranged_length, ranged_md5, ranged_ranges = download(ranged_path, threshold="1MB", chunksize="1MB")

assert whole_ranges == 0, f"the single-GET control issued {whole_ranges} Range header(s) -- it is not a control"
assert ranged_ranges >= 3, (
    f"the ranged arm issued only {ranged_ranges} Range header(s) -- `multipart_threshold` did not take "
    f"effect, so the md5 comparison below would be comparing two identical plain GETs."
)

assert whole_length == expected_length, f"single-GET size {whole_length} != ContentLength {expected_length}"
assert ranged_length == expected_length, f"ranged size {ranged_length} != ContentLength {expected_length}"
assert whole_md5 == ranged_md5, (
    f"ranged download does not match the single-GET control: {ranged_md5} != {whole_md5}. Same byte "
    f"count, different content -- this is exactly the reassembly bug the A/B exists to catch."
)
# This object was uploaded whole, so its ETag IS the md5 (no `-N` suffix). Free third opinion; skipped
# rather than asserted if AWS ever re-uploads it multipart, because then the ETag is an md5-of-md5s.
etag = head["ETag"].strip('"')
if "-" not in etag:
    assert etag == whole_md5, f"md5 {whole_md5} != ETag {etag} -- the bytes are not what S3 stored"
    print(f"  ETag matches md5 ({etag})")
print(f"  ranged download: {expected_length:,} B, md5 identical across {whole_ranges}-range control "
      f"and {ranged_ranges}-range split  ok")